# 跑yolo模型

## 第一步：环境配置

**1.GPU检查**

代码：！是执行系统shell命令前缀

结果：

分配T4 GPU，检查GPU是否分配成功

Perf表示性能状态，P0最高，p12最低。

memory-usage:显存使用情况

GPU-util:GPU利用率

In [4]:
!nvidia-smi

Mon May 25 13:45:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**2.挂云盘**：这是colab专门提供和google drive交互的模块

mount是挂载的意思，挂载到xx文件夹  

打开左侧文件可以看到，colab上的文件，关机后会消失，而drive/mydrive文件夹的内容
是保存到自己电脑云端的

colab本地运行的更快，需要频繁读写的文件如克隆的仓库和数据集都要放到colab  
放到drive的文件：训练好的.pt，导出的历史轨迹.csv等等

In [5]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


**3.安装所有必要的库**

yolo的库

获取数据集文件的库

卡尔曼滤波的库

In [6]:
!pip install ultralytics roboflow filterpy

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.0/178.0 kB 20.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 77.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 113.8 MB/s eta 0:00:00
  Created wheel for filterpy: filename=filterpy-1.4.5-py3-none-any.whl size=110460 sha256=8206eb5d283d4535f7f179b63a150ff295d5b79a5bb02b99261653b5c9cac876
  Stored in directory: /root/.cache/pip/wheels/77/bf/4c/b0c3f4798a0166668752312a67118b27a3cd341e13ac0ae6ee
Successfully built filterpy
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    U

**4.创建目录**

将所有的代码和配置都放在 /content/autonomous_driving 目录下

In [7]:
import os

PROJECT_PATH = "/content/autonomous_driving"
for folder in ["configs", "src", "weights", "data", "results"]:
    os.makedirs(os.path.join(PROJECT_PATH, folder), exist_ok=True)

print(f"✅ 项目目录 {PROJECT_PATH} 已建立！")

✅ 项目目录 /content/autonomous_driving 已建立！


## 第二步：获取并处理数据集

**5.获取数据集**

出现的问题：本地下载的数据集文件在google云盘下载的太慢且网络波动后断掉了。

解决方案：用别人提供的文件。

找了个Roboflow平台（计算机视觉数据管理预处理）

在其中找到kitti数据集下载下来

In [ ]:
%cd /content/autonomous_driving/data

from roboflow import Roboflow
rf = Roboflow(api_key="nejPKc5WYocno3KDVt6j")
project = rf.workspace("sudip-dhakal").project("kitti-uajb1")
version = project.version(1)
dataset = version.download("yolov8")



#  检查并重命名
# 下载后的文件夹通常叫 "KITTI-1"，我们把它改名为 "kitti_raw"
import os
if os.path.exists("/content/autonomous_driving/data/KITTI-1"):
    os.rename("/content/autonomous_driving/data/KITTI-1", "/content/autonomous_driving/data/kitti_raw")
    print("✅ 数据集已成功下载并重命名为 kitti_raw")
else:
    # 有时候名字里带空格或版本号，如果改名失败，我们打印一下当前目录看看它叫什么
    print("当前目录下的文件夹有：", os.listdir())

/content/autonomous_driving/data
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to KITTI-1 in yolov8:: 100%|██████████| 14968/14968 [00:03<00:00, 4444.57it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
✅ 数据集已成功下载并重命名为 kitti_raw


In [ ]:
# 看看训练集里到底有多少张图片
!ls /content/autonomous_driving/data/kitti_raw/train/images | wc -l

7481


**6.划分数据集与验证集**

In [ ]:
import random
import shutil

train_img_path = "/content/autonomous_driving/data/kitti_raw/train/images"
train_lab_path = "/content/autonomous_driving/data/kitti_raw/train/labels"
valid_img_path = "/content/autonomous_driving/data/kitti_raw/valid/images"
valid_lab_path = "/content/autonomous_driving/data/kitti_raw/valid/labels"

os.makedirs(valid_img_path, exist_ok=True)
os.makedirs(valid_lab_path, exist_ok=True)

all_images = [f for f in os.listdir(train_img_path) if f.endswith('.jpg')]
val_count = int(len(all_images) * 0.1)
#列表随机抽取
val_images = random.sample(all_images, val_count)

for img_name in val_images:
    #移动图片
    shutil.move(os.path.join(train_img_path, img_name), os.path.join(valid_img_path, img_name))
    lab_name = img_name.replace('.jpg', '.txt')
    if os.path.exists(os.path.join(train_lab_path, lab_name)):
        shutil.move(os.path.join(train_lab_path, lab_name), os.path.join(valid_lab_path, lab_name))

print(f"✅ 数据集重新划分完成，验证集现在有 {len(val_images)} 张图。")

✅ 数据集重新划分完成，验证集现在有 748 张图。


In [ ]:
# 统计训练集图片数量 (预期应该是 7481 - 748 = 6733 左右)
print("训练集图片数：")
!ls /content/autonomous_driving/data/kitti_raw/train/images | wc -l

# 统计验证集图片数量 (预期应该是 748)
print("验证集图片数：")
!ls /content/autonomous_driving/data/kitti_raw/valid/images | wc -l

训练集图片数：
6733
验证集图片数：
748


## 第三步：训练

**7.生成yaml文件**

In [ ]:
import yaml

data_config = {
    'path': '/content/autonomous_driving/data/kitti_raw', # 绝对路径
    'train': 'train/images',
    'val': 'valid/images',
    'nc': 2,
    'names': ['car', 'person']
}

with open('/content/autonomous_driving/configs/kitti_config.yaml', 'w') as f:
    yaml.dump(data_config, f)

print("✅ 配置文件已生成：/content/autonomous_driving/configs/kitti_config.yaml")

✅ 配置文件已生成：/content/autonomous_driving/configs/kitti_config.yaml


**8.正式训练**

In [ ]:
from ultralytics import YOLO

# 加载模型
model = YOLO("yolov8s.pt")

# 启动训练
results = model.train(
    data="/content/autonomous_driving/configs/kitti_config.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=8,
    project="/content/autonomous_driving/results",
    name="kitti_v8s_full"
)

Ultralytics 8.4.53 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/autonomous_driving/configs/kitti_config.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=kitti_v8s_full, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, ov

**9.保存下来**

In [ ]:
import shutil
import os

# 定义云盘备份路径
backup_dir = "/content/drive/MyDrive/Lantu_Project_Backup"
os.makedirs(backup_dir, exist_ok=True)

# 1. 备份最好的权重
shutil.copy("/content/autonomous_driving/results/kitti_v8s_full/weights/best.pt", f"{backup_dir}/best.pt")

# 2. 备份训练指标图
shutil.copy("/content/autonomous_driving/results/kitti_v8s_full/results.png", f"{backup_dir}/results.png")
shutil.copy("/content/autonomous_driving/results/kitti_v8s_full/confusion_matrix.png", f"{backup_dir}/confusion_matrix.png")

print(f"✅ 所有的‘宝贝’都已安全备份到云盘：{backup_dir}")

✅ 所有的‘宝贝’都已安全备份到云盘：/content/drive/MyDrive/Lantu_Project_Backup


# 感知


## 环境与装载best.pt

colab关闭之后，可以重新

In [8]:
from ultralytics import YOLO
# 重点：路径直接指向云盘里的那个备份
model = YOLO("/content/drive/MyDrive/YOLO_Project/best.pt")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [15]:
import cv2
import numpy as np
from ultralytics import YOLO
from collections import defaultdict
import os

# 1. 确认模型路径 (请确保这个路径指向你云盘里的 best.pt)
# 如果你之前存的文件夹名字不一样，请修改这里！
model_path = "/content/drive/MyDrive/YOLO_Project/best.pt"
model = YOLO(model_path)

# 2. 视频输入与输出路径
video_path = "/content/autonomous_driving/data/my_video.mp4"
temp_out = '/content/autonomous_driving/results/temp_output.mp4'
final_out = '/content/autonomous_driving/results/Lantu_Final_Prediction.mp4'

# 检查视频是否存在
if not os.path.exists(video_path):
    print(f"❌ 找不到视频，请确认你是否将视频重命名为了 my_video.mp4 并放在了 {video_path} 目录下！")
else:
    cap = cv2.VideoCapture(video_path)
    w, h, fps = int(cap.get(3)), int(cap.get(4)), int(cap.get(5))
    out = cv2.VideoWriter(temp_out, cv2.VideoWriter_fourcc(*'mp4v'), fps, (w, h))
    track_history = defaultdict(lambda: [])

    print("🚀 正在你的专属视频上生成预测轨迹，请稍候...")
    frame_count = 0

    while cap.isOpened():
        success, frame = cap.read()
        if not success:
            break

        # 运行目标跟踪，conf=0.15 过滤低置信度噪点
        results = model.track(frame, persist=True, tracker="botsort.yaml", conf=0.15, verbose=False)

        if results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            xywh = results[0].boxes.xywh.cpu().numpy()
            track_ids = results[0].boxes.id.int().cpu().tolist()
            clss = results[0].boxes.cls.int().cpu().tolist()

            for box, center_box, track_id, cls in zip(boxes, xywh, track_ids, clss):
                x1, y1, x2, y2 = box
                cx, cy, bw, bh = center_box

                # 1. 绘制检测框与ID (科技蓝)
                cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), (255, 150, 0), 2)
                cv2.putText(frame, f"ID:{track_id}", (int(x1), int(y1) - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 255), 2)

                # 2. 记录并绘制历史轨迹 (红色)
                track = track_history[track_id]
                track.append((float(cx), float(cy)))
                if len(track) > 30: track.pop(0) # 最多保留30帧尾巴

                if len(track) > 2:
                    pts = np.hstack(track).astype(np.int32).reshape((-1, 1, 2))
                    cv2.polylines(frame, [pts], isClosed=False, color=(0, 0, 255), thickness=2)

                # 3. 🎯 核心卖点：绿色预测箭头 🎯
                # 收集到 5 帧以上的数据就开始预测 (反应更灵敏)
                if len(track) >= 5:
                    # 计算最近 5 帧的平均速度矢量
                    dx = (track[-1][0] - track[-5][0]) / 4
                    dy = (track[-1][1] - track[-5][1]) / 4
                    # 预测未来 20 帧的位置 (相当于预测约 0.7 秒后的落点)
                    # 乘数越大，绿线越长
                    pred_x = track[-1][0] + dx * 20
                    pred_y = track[-1][1] + dy * 20

                    # 画绿色带箭头的预测线
                    cv2.arrowedLine(frame, (int(cx), int(cy)), (int(pred_x), int(pred_y)),
                                    (0, 255, 0), 3, tipLength=0.3)

        out.write(frame)
        frame_count += 1
        if frame_count % 50 == 0:
            print(f"⏳ 已处理 {frame_count} 帧...")

    cap.release()
    out.release()

    # 4. 使用 FFmpeg 强制转码，保证在 Windows/Mac 上绝对能播放
    print("⚙️ 正在进行 H.264 工业级转码...")
    os.system(f"ffmpeg -y -i {temp_out} -vcodec libx264 -pix_fmt yuv420p {final_out}")
    print(f"✅ 完美搞定！请下载最终演示视频：{final_out}")

🚀 正在你的专属视频上生成预测轨迹，请稍候...
⏳ 已处理 50 帧...
⏳ 已处理 100 帧...
⏳ 已处理 150 帧...
⏳ 已处理 200 帧...
⏳ 已处理 250 帧...
⏳ 已处理 300 帧...
⏳ 已处理 350 帧...
⏳ 已处理 400 帧...
⏳ 已处理 450 帧...
⏳ 已处理 500 帧...
⏳ 已处理 550 帧...
⏳ 已处理 600 帧...
⏳ 已处理 650 帧...
⏳ 已处理 700 帧...
⏳ 已处理 750 帧...
⏳ 已处理 800 帧...
⏳ 已处理 850 帧...
⚙️ 正在进行 H.264 工业级转码...
✅ 完美搞定！请下载最终演示视频：/content/autonomous_driving/results/Lantu_Final_Prediction.mp4


In [16]:
import shutil

# 将刚才生成的终极视频，直接拷贝到你的谷歌云盘根目录
source = '/content/autonomous_driving/results/Lantu_Final_Prediction.mp4'
destination = '/content/drive/MyDrive/Final_Prediction_Demo.mp4'

shutil.copy(source, destination)
print("✅ 视频已成功瞬间转移到你的谷歌云盘！")

✅ 视频已成功瞬间转移到你的谷歌云盘！
